# 1. A base escolhida

A base utilizada foi o arquivo **`loja_roupas.csv`**, a nova base disponibilizada no projeto. Ela registra vendas de uma loja de roupas e foi escolhida por conter informações suficientes para analisar o comportamento das vendas sob diferentes perspectivas: período, categoria, produto, gênero do produto, quantidade, preço unitário, valor total, forma de pagamento, cadastro do cliente e estado.
Essa base é adequada ao desafio porque permite investigar quais categorias e produtos concentram as vendas, a quantidade de itens por venda, o preço unitário, o valor total, as formas de pagamento, a participação de clientes cadastrados, o gênero dos produtos e a variação das vendas ao longo do tempo e por estado. Além disso, o campo `valor` é consistente com o cálculo `quantidade × preco_unitario` em todos os registros. Como limitação, o arquivo não contém identificador individual do cliente, descontos, estoque ou informações de entrega.

## Instalando o Pandas

In [39]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [40]:
# Carregando a nova base e vendo os tipos de dados das colunas
import pandas as pd

df = pd.read_csv("data/loja_roupas.csv", parse_dates=["data"])
df.dtypes


id_venda                       int64
data                  datetime64[us]
categoria                        str
produto                          str
genero_produto                   str
quantidade                     int64
preco_unitario               float64
valor                        float64
forma_pagamento                  str
cliente_cadastrado               str
estado                           str
dtype: object

# 2. Dicionário de dados

O dicionário abaixo descreve as colunas da nova base `loja_roupas.csv`, seus tipos e a função de cada campo na análise.

| Coluna | Tipo no pandas | Descrição | Valores ou observações |
|---|---|---|---|
| `id_venda` | `int64` | Identificador único da venda | 509 valores distintos; não deve ser usado como medida |
| `data` | `datetime64[us]` | Momento em que a venda foi realizada | De 01/03/2025 a 20/06/2025 |
| `categoria` | `str` | Categoria de roupa ou acessório vendido | Camisas, Calças, Vestidos e Acessórios |
| `produto` | `str` | Nome do produto vendido | 16 produtos, como Camisa polo, Bolsa e Vestido casual |
| `genero_produto` | `str` | Gênero ao qual o produto é direcionado | Unissex, Feminino ou Masculino |
| `quantidade` | `int64` | Número de unidades do produto na venda | De 1 a 4 unidades; total de 753 itens |
| `preco_unitario` | `float64` | Preço de uma unidade do produto, em R$ | De R$ 35,60 a R$ 180,10 |
| `valor` | `float64` | Valor total registrado para a venda, em R$ | De R$ 35,85 a R$ 705,20; corresponde a `quantidade × preco_unitario` |
| `forma_pagamento` | `str` | Meio utilizado para pagar a compra | Pix, Cartão Crédito, Cartão Débito ou Dinheiro |
| `cliente_cadastrado` | `str` | Indica se o cliente possuía cadastro na loja | `Sim` ou `Não` |
| `estado` | `str` | Estado brasileiro associado à venda | 27 UFs, incluindo o Distrito Federal |

# 3. Perguntas que orientam a análise

As perguntas abaixo definem os principais pontos de investigação da análise exploratória e relacionam cada questão ao seu objetivo de negócio.

| Pergunta | Objetivo |
|---|---|
| Qual é o faturamento total e o ticket médio das vendas? | Avaliar o desempenho financeiro e o valor médio gasto por venda. |
| Quais categorias e produtos geram maior faturamento? | Identificar os itens mais relevantes para a receita da loja. |
| Como o faturamento e a quantidade de vendas evoluem ao longo dos meses? | Identificar tendências de crescimento ou queda nas vendas. |
| Quais horários concentram o maior volume de vendas? | Apoiar decisões sobre o funcionamento e a distribuição dos funcionários. |
| Vendas de clientes cadastrados possuem ticket médio maior que vendas de clientes não cadastrados? | Testar a hipótese de que o cadastro está associado a um maior valor médio por venda. |

## Transformação e limpeza de dados

Para a limpeza e transformação de dados, foi feita uma analise exploratória para identificar quais pontos precisam ser limpados e transformados.

#### Limpeza
- Campos como categoria, produto, genero_produto, forma_pagamento, cliente_cadastrado, com os campos despadronizados(letras em minusculas com somente a primeira letra em maiúscula). Colocar os dados todos em maiúsculas

#### Transformação
- Separar os campos de dia, mes, ano e hora do csv criando esses campos com o objetivo de facilitar as analises

#### Analise da geral do CSV

In [41]:
print(f"Quantidade de linhas: {df.shape[0]}")
print(f"Quantidade de colunas: {df.shape[1]}")
print(f"Quantidade de valores nulos: {df.isnull().sum().sum()}")

Quantidade de linhas: 509
Quantidade de colunas: 11
Quantidade de valores nulos: 0


## Transformação e limpeza de dados

Para a limpeza e transformação de dados, foi feita uma analise exploratória para identificar quais pontos precisam ser limpados e transformados.

#### Limpeza
- Campos como categoria, produto, genero_produto, forma_pagamento, cliente_cadastrado, com os campos despadronizados(letras em minusculas com somente a primeira letra em maiúscula). Colocar os dados todos em maiúsculas

#### Transformação
- Separar os campos de dia, mes, ano e hora do csv criando esses campos com o objetivo de facilitar as analises

#### Campos para maiúsculas(limpeza)

In [42]:
df["categoria"] = df["categoria"].str.upper()
df["produto"] = df["produto"].str.upper()
df["genero_produto"] = df["genero_produto"].str.upper()
df["forma_pagamento"] = df["forma_pagamento"].str.upper()
df["cliente_cadastrado"] = df["cliente_cadastrado"].str.upper()

display(df.head(10))

,id_venda,data,categoria,produto,genero_produto,quantidade,preco_unitario,valor,forma_pagamento,cliente_cadastrado,estado
0,1,2025-03-01 09:45:27,CAMISAS,CAMISA POLO,MASCULINO,1,86.16,86.16,PIX,NÃO,SE
1,2,2025-03-01 14:27:04,CALÇAS,CALÇA DE MOLETOM,UNISSEX,1,144.66,144.66,CARTÃO CRÉDITO,SIM,MT
2,3,2025-03-01 19:54:07,CAMISAS,CAMISA SOCIAL,UNISSEX,1,76.46,76.46,CARTÃO DÉBITO,NÃO,MA
3,4,2025-03-01 19:26:20,ACESSÓRIOS,BONÉ,UNISSEX,1,43.83,43.83,DINHEIRO,NÃO,MA
4,5,2025-03-01 12:26:42,CAMISAS,CAMISA POLO,MASCULINO,2,90.45,180.90,CARTÃO CRÉDITO,SIM,RO
5,6,2025-03-02 18:36:19,VESTIDOS,VESTIDO CASUAL,FEMININO,2,142.62,285.24,PIX,NÃO,TO
6,7,2025-03-02 13:18:13,CAMISAS,CAMISA SOCIAL,UNISSEX,3,68.43,205.29,CARTÃO DÉBITO,NÃO,RJ
7,8,2025-03-02 16:28:28,CALÇAS,CALÇA SOCIAL,UNISSEX,1,122.78,122.78,CARTÃO CRÉDITO,NÃO,PE
8,9,2025-03-02 11:42:05,ACESSÓRIOS,BOLSA,FEMININO,1,38.55,38.55,DINHEIRO,SIM,RO
9,10,2025-03-03 10:52:48,CAMISAS,CAMISA POLO,MASCULINO,1,71.20,71.20,PIX,NÃO,MA


#### Tranformação

In [43]:
df["dia"] = df["data"].dt.day
df["mes"] = df["data"].dt.month
df["ano"] = df["data"].dt.year
df["horario"] = df["data"].dt.time

display(df.head(10))

,id_venda,data,categoria,produto,genero_produto,quantidade,preco_unitario,valor,forma_pagamento,cliente_cadastrado,estado,dia,mes,ano,horario
0,1,2025-03-01 09:45:27,CAMISAS,CAMISA POLO,MASCULINO,1,86.16,86.16,PIX,NÃO,SE,1,3,2025,09:45:27
1,2,2025-03-01 14:27:04,CALÇAS,CALÇA DE MOLETOM,UNISSEX,1,144.66,144.66,CARTÃO CRÉDITO,SIM,MT,1,3,2025,14:27:04
2,3,2025-03-01 19:54:07,CAMISAS,CAMISA SOCIAL,UNISSEX,1,76.46,76.46,CARTÃO DÉBITO,NÃO,MA,1,3,2025,19:54:07
3,4,2025-03-01 19:26:20,ACESSÓRIOS,BONÉ,UNISSEX,1,43.83,43.83,DINHEIRO,NÃO,MA,1,3,2025,19:26:20
4,5,2025-03-01 12:26:42,CAMISAS,CAMISA POLO,MASCULINO,2,90.45,180.90,CARTÃO CRÉDITO,SIM,RO,1,3,2025,12:26:42
5,6,2025-03-02 18:36:19,VESTIDOS,VESTIDO CASUAL,FEMININO,2,142.62,285.24,PIX,NÃO,TO,2,3,2025,18:36:19
6,7,2025-03-02 13:18:13,CAMISAS,CAMISA SOCIAL,UNISSEX,3,68.43,205.29,CARTÃO DÉBITO,NÃO,RJ,2,3,2025,13:18:13
7,8,2025-03-02 16:28:28,CALÇAS,CALÇA SOCIAL,UNISSEX,1,122.78,122.78,CARTÃO CRÉDITO,NÃO,PE,2,3,2025,16:28:28
8,9,2025-03-02 11:42:05,ACESSÓRIOS,BOLSA,FEMININO,1,38.55,38.55,DINHEIRO,SIM,RO,2,3,2025,11:42:05
9,10,2025-03-03 10:52:48,CAMISAS,CAMISA POLO,MASCULINO,1,71.20,71.20,PIX,NÃO,MA,3,3,2025,10:52:48


#### Gerando o csv limpo e transformado

In [44]:
df.to_csv("data/loja_roupas_tratado_limpo.csv", index=False, sep=",", encoding="utf-8")